%md
This project is an end-to-end data engineering pipeline using Pyspark, Databricks, and Delta lake to process the data.

The pipelines follows the Medallion architecture Bronze --> It stores raw data in delta format, Silver --> It applied data type comversions, standarizing the data, data quality validation and Gold --> It used for analytics ready tables for inspection & failure trens by High, Medium and Low risk categories and year.

Source: Food Inspection from City of Chicago which has 316703 raw records, 17 original columns.

Tech Stack: Python,Pyspark,Databricks,Delta lake,Spark SQL

Flow of Pipeline: Raw CSV --> Bronze Data Table --> Pyspark Cleaning & Validation --> Silver Delta Table --> Gold Analytics Tables --> Data Reconciliation & Validation

#Data Engineering Features 
- Schema and Data type conversion
- Data standardization
- Data-quality validation
- City-name correction rules
- Zip-code validation
- Geographic-coordinate validation
- Record-level quality flags
- Delta Lake storage
- Medallion architecture
- End-to-End data reconciliation

In [0]:
from pyspark.sql import functions as F

In [0]:
from pyspark.sql import functions as F

bronze_df = spark.table(
    "workspace.default.chicago_food_inspections_bronze"
)

print("Bronze Records:", bronze_df.count())
print("Bronze Columns:", len(bronze_df.columns))

bronze_df.printSchema()

In [0]:
from pyspark.sql import functions as F

silver_df = (
    bronze_df
    .withColumn(
        "inspection_id",
        F.col("inspection_id").cast("long")
    )
    .withColumn(
        "inspection_date",
        F.to_date(
            F.col("inspection_date"),
            "MM/dd/yyyy"
        )
    )
    .withColumn(
        "latitude",
        F.col("latitude").cast("double")
    )
    .withColumn(
        "longitude",
        F.col("longitude").cast("double")
    )
)

silver_df.select(
    "inspection_id",
    "inspection_date",
    "latitude",
    "longitude"
).show(5)

silver_df.printSchema()

In [0]:
from pyspark.sql import functions as F

# Standardize city and state formatting
silver_df = (
    silver_df
    .withColumn("city_raw", F.col("city"))
    .withColumn("city", F.upper(F.trim(F.col("city"))))
    .withColumn("state", F.upper(F.trim(F.col("state"))))
)

# Known city-name corrections
city_corrections = {
    "CCHICAGO": "CHICAGO",
    "CHICAGOCHICAGO": "CHICAGO",
    "CHICAGOO": "CHICAGO",
    "CHICAGO.": "CHICAGO",
    "312CHICAGO": "CHICAGO",
    "CHCHICAGO": "CHICAGO",
    "MERRIVILLE": "MERRILLVILLE",
    "NILES NILES": "NILES"
}

mapping = F.create_map(
    *[
        item
        for old, new in city_corrections.items()
        for item in (F.lit(old), F.lit(new))
    ]
)

silver_df = silver_df.withColumn(
    "city",
    F.coalesce(
        F.element_at(mapping, F.col("city")),
        F.col("city")
    )
)

# Check cleaning results
silver_df.groupBy("city") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(15, truncate=False)

In [0]:
from pyspark.sql import functions as F

# City quality validation
silver_df = silver_df.withColumn(
    "city_quality_flag",

    F.when(
        F.col("city").isNull() |
        (F.trim(F.col("city")) == ""),
        "MISSING_CITY"
    )
    .when(
        F.col("city") == "INACTIVE",
        "INVALID_CITY"
    )
    .otherwise("NOT_FLAGGED")
)

# State quality validation
silver_df = silver_df.withColumn(
    "state_quality_flag",

    F.when(
        F.col("state").isNull() |
        (F.trim(F.col("state")) == ""),
        "MISSING_STATE"
    )
    .when(
        (F.col("zip") == "60642") &
        (F.col("state") != "IL"),
        "STATE_ZIP_MISMATCH"
    )
    .when(
        F.col("state") != "IL",
        "OUT_OF_STATE_REVIEW"
    )
    .otherwise("NOT_FLAGGED")
)

# Validate city flags
silver_df.groupBy("city_quality_flag") \
    .count().show()

# Validate state flags
silver_df.groupBy("state_quality_flag") \
    .count().show()

In [0]:
from pyspark.sql import functions as F

# ZIP code validation
silver_df = silver_df.withColumn(
    "zip_quality_flag",

    F.when(
        F.col("zip").isNull() |
        (F.trim(F.col("zip")) == ""),
        "MISSING_ZIP"
    )
    .when(
        ~F.trim(F.col("zip")).rlike(
            r"^[0-9]{5}(-[0-9]{4})?$"
        ),
        "INVALID_ZIP_FORMAT"
    )
    .otherwise("NOT_FLAGGED")
)

# Coordinate validation
silver_df = silver_df.withColumn(
    "coordinate_quality_flag",

    F.when(
        F.col("latitude").isNull() &
        F.col("longitude").isNull(),
        "MISSING_COORDINATES"
    )
    .when(
        F.col("latitude").isNull() |
        F.col("longitude").isNull(),
        "PARTIAL_COORDINATES"
    )
    .when(
        ~F.col("latitude").between(-90, 90) |
        ~F.col("longitude").between(-180, 180),
        "INVALID_COORDINATES"
    )
    .otherwise("NOT_FLAGGED")
)

# Check results
silver_df.groupBy("zip_quality_flag").count().show()

silver_df.groupBy("coordinate_quality_flag").count().show()

In [0]:
from pyspark.sql import functions as F

silver_df = silver_df.withColumn(
    "overall_quality_status",

    F.when(
        (F.col("city_quality_flag") != "NOT_FLAGGED") |
        (F.col("state_quality_flag") != "NOT_FLAGGED") |
        (F.col("zip_quality_flag") != "NOT_FLAGGED") |
        (
            F.col("coordinate_quality_flag").isin(
                "INVALID_COORDINATES",
                "PARTIAL_COORDINATES"
            )
        ),
        "REVIEW_REQUIRED"
    )

    .when(
        F.col("coordinate_quality_flag") ==
        "MISSING_COORDINATES",
        "USABLE_WITHOUT_GEO"
    )

    .otherwise("NOT_FLAGGED")
)

silver_df.groupBy("overall_quality_status") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

In [0]:
# Validate before saving
assert silver_df.count() == bronze_df.count()
assert silver_df.select("inspection_id").distinct().count() == 316703
assert silver_df.filter(
    F.col("inspection_id").isNull() |
    F.col("inspection_date").isNull()
).count() == 0

# Write the final Silver table
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "workspace.default.chicago_food_inspections_silver"
    )

print("Final Silver Delta table updated successfully!")

In [0]:
from pyspark.sql import functions as F

# Read finalized Silver Delta table
silver_table = spark.table(
    "workspace.default.chicago_food_inspections_silver"
)

# Calculate risk-based inspection metrics
gold_risk_df = (
    silver_table
    .filter(
        F.col("results").isin(
            "Pass", "Fail", "Pass w/ Conditions"
        )
    )
    .filter(
        F.col("risk").isin(
            "Risk 1 (High)",
            "Risk 2 (Medium)",
            "Risk 3 (Low)"
        )
    )
    .groupBy("risk")
    .agg(
        F.count("*").alias("total_inspections"),

        F.sum(
            F.when(F.col("results") == "Fail", 1)
            .otherwise(0)
        ).alias("failed_inspections")
    )
    .withColumn(
        "failure_rate_pct",
        F.round(
            F.col("failed_inspections") * 100 /
            F.col("total_inspections"),
            2
        )
    )
)

# Save as a Delta table
gold_risk_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "workspace.default.chicago_inspection_risk_gold"
    )

# Verify saved results
display(
    spark.table(
        "workspace.default.chicago_inspection_risk_gold"
    )
)

In [0]:
from pyspark.sql import functions as F

gold_yearly_df = (
    silver_table
    .filter(
        F.col("results").isin(
            "Pass",
            "Fail",
            "Pass w/ Conditions"
        )
    )
    .withColumn(
        "inspection_year",
        F.year(F.col("inspection_date"))
    )
    .groupBy("inspection_year")
    .agg(
        F.count("*").alias("total_inspections"),

        F.sum(
            F.when(F.col("results") == "Fail", 1)
            .otherwise(0)
        ).alias("failed_inspections")
    )
    .withColumn(
        "failure_rate_pct",
        F.round(
            F.col("failed_inspections") * 100 /
            F.col("total_inspections"),
            2
        )
    )
    .orderBy("inspection_year")
)

# Save Gold table
gold_yearly_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "workspace.default.chicago_inspection_yearly_gold"
    )

print("Yearly Gold table updated successfully!")

yearly_table = spark.table(
    "workspace.default.chicago_inspection_yearly_gold"
)

print("Total Years:", yearly_table.count())

yearly_table.agg(
    F.sum("total_inspections").alias("total_inspections"),
    F.sum("failed_inspections").alias("failed_inspections")
).show()

In [0]:
from pyspark.sql import functions as F

bronze = spark.table(
    "workspace.default.chicago_food_inspections_bronze"
)

silver = spark.table(
    "workspace.default.chicago_food_inspections_silver"
)

gold_risk = spark.table(
    "workspace.default.chicago_inspection_risk_gold"
)

gold_yearly = spark.table(
    "workspace.default.chicago_inspection_yearly_gold"
)

# Bronze vs Silver record count
bronze_count = bronze.count()
silver_count = silver.count()

assert bronze_count == silver_count

# Inspection ID uniqueness
assert (
    silver.select("inspection_id")
    .distinct()
    .count()
    == silver_count
)

# Completed inspection records used for yearly Gold
completed = silver.filter(
    F.col("results").isin(
        "Pass",
        "Fail",
        "Pass w/ Conditions"
    )
)

# Validate yearly Gold totals
yearly_totals = gold_yearly.agg(
    F.sum("total_inspections").alias("total"),
    F.sum("failed_inspections").alias("failed")
).first()

assert yearly_totals["total"] == completed.count()

assert yearly_totals["failed"] == completed.filter(
    F.col("results") == "Fail"
).count()

# Records eligible for risk Gold
risk_filtered = completed.filter(
    F.col("risk").isin(
        "Risk 1 (High)",
        "Risk 2 (Medium)",
        "Risk 3 (Low)"
    )
)

# Validate risk Gold totals
risk_totals = gold_risk.agg(
    F.sum("total_inspections").alias("total"),
    F.sum("failed_inspections").alias("failed")
).first()

assert risk_totals["total"] == risk_filtered.count()

assert risk_totals["failed"] == risk_filtered.filter(
    F.col("results") == "Fail"
).count()

print("Bronze Records:", bronze_count)
print("Silver Records:", silver_count)
print("Yearly Gold Records:", gold_yearly.count())
print("Completed Inspections:", yearly_totals["total"])
print("Failed Inspections:", yearly_totals["failed"])
print("Risk Gold Records:", gold_risk.count())
print("Risk-Eligible Inspections:", risk_totals["total"])

print("\nAll reconciliation checks passed!")

In [0]:
from pyspark.sql import functions as F

print("=" * 60)
print("CHICAGO FOOD INSPECTIONS ETL PIPELINE - FINAL SUMMARY")
print("=" * 60)

print(f"Bronze Records              : {bronze.count():,}")
print(f"Silver Records              : {silver.count():,}")
print(f"Silver Columns              : {len(silver.columns)}")

print("\nDATA QUALITY STATUS")
silver.groupBy("overall_quality_status") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(truncate=False)

print("GOLD LAYER")
print(f"Yearly Gold Rows            : {gold_yearly.count()}")
print(f"Risk Gold Rows              : {gold_risk.count()}")
print(f"Completed Inspections       : {completed.count():,}")
print(f"Failed Inspections          : {completed.filter(F.col('results') == 'Fail').count():,}")
print(f"Risk-Eligible Inspections   : {risk_filtered.count():,}")

print("\nPIPELINE STATUS: SUCCESS")
print("All Bronze, Silver and Gold reconciliation checks passed.")